# Main Preprocess file for Round-Robin assignment Replication

In [1]:
import numpy as np
import pandas as pd

In [2]:
csv_path = "../../data/real_bitcoin_blocks_raw.csv"
df = pd.read_csv(csv_path)

df.tail()

,number,timestamp,size,transaction_count,bits,block_number,total_output_satoshis,total_output_satoshis_excl_coinbase,total_fee_satoshis,tx_count_check,difficulty
810904,810904,2023-10-06 12:36:45+00:00,1471359,2641,1704e90f,810904,7.607876e+11,7.601375e+11,25173010.0,2641,5.732151e+13
810905,810905,2023-10-06 12:45:49+00:00,1427754,2288,1704e90f,810905,1.161340e+12,1.160692e+12,23663392.0,2288,5.732151e+13
810906,810906,2023-10-06 12:50:15+00:00,1612966,2016,1704e90f,810906,4.643628e+11,4.637224e+11,15483511.0,2016,5.732151e+13
810907,810907,2023-10-06 13:37:00+00:00,1486563,3496,1704e90f,810907,2.469789e+12,2.469101e+12,63223285.0,3496,5.732151e+13
810908,810908,2023-10-06 13:37:21+00:00,1647105,3397,1704e90f,810908,5.116429e+11,5.109731e+11,44811365.0,3397,5.732151e+13


In [3]:
# Check where values equal 0, then check across rows
rows_with_zero = df[(df[['total_output_satoshis']] == 0).any(axis=1)]
rows_with_zero

,number,timestamp,size,transaction_count,bits,block_number,total_output_satoshis,total_output_satoshis_excl_coinbase,total_fee_satoshis,tx_count_check,difficulty
501726,501726,2017-12-30 12:55:20+00:00,200,1,18009645,501726,0.0,0.0,0.0,1,1.873105e+12


In [4]:
for col in ["total_output_satoshis", "total_output_satoshis_excl_coinbase"]:
    zero_volume = df[df[col] == 0]
    print(col, "zero-volume blocks:", len(zero_volume))
    print(zero_volume["number"].describe())  # are they clustered early, or scattered?

total_output_satoshis zero-volume blocks: 1
count         1.0
mean     501726.0
std           NaN
min      501726.0
25%      501726.0
50%      501726.0
75%      501726.0
max      501726.0
Name: number, dtype: float64
total_output_satoshis_excl_coinbase zero-volume blocks: 89548
count     89548.000000
mean      72387.225075
std      105262.128796
min           0.000000
25%       22501.750000
50%       45156.500000
75%       75721.250000
max      810811.000000
Name: number, dtype: float64


In [5]:
def clean_zero_volume(df, column, patch=True):
    """
    patch=True  -> replace any 0 values in `column` with the column median (like Ethan's missing-value handling)
    patch=False -> leave the data exactly as it is now (current behavior)
    """
    df_cleaned = df.copy()
    if patch:
        df_cleaned[column] = df_cleaned[column].replace(0, np.nan)
        df_cleaned[column] = df_cleaned[column].fillna(df_cleaned[column].median())
        print(f"Patched {(df[column] == 0).sum()} zero-value row(s) in '{column}'")
    return df_cleaned

In [6]:
df = clean_zero_volume(df=df, column='total_output_satoshis', patch=True)

Patched 1 zero-value row(s) in 'total_output_satoshis'


In [7]:
# Either "total_output_satoshis" or "total_output_satoshis_excl_coinbase"
# Either "full" (all 9 features) or "reduced" (drop avg_transactions and avg_volume)

# Definition of Efficiency
# Either ratio_of_means: (avg_transaction / avg_volume)
# Or     mean_of_ratios: average of (n_transactions / transaction_volume) per block

# Definition of Age
# Method 1              ("distance_from_tip"): gap between each miner's last block and the very last block in the whole dataset
# Method 2 (Ethan's)    ("neighbor_gap")     : sort miners by last-mined time, then gap to the miner right before you

def preprocess(df, volume_column, efficiency_definition, age_definition):
    # Rename Columns
    processed_df = df.rename(columns={
        'number': 'block_id',
        'transaction_count': 'n_transactions',
        volume_column: 'transaction_volume',
    })

    # Drop Unwanted Columns
    if volume_column == "total_output_satoshis":
        drop_columns = ['total_output_satoshis_excl_coinbase', 'total_fee_satoshis', 'block_number', 'tx_count_check', 'bits']
        
    elif volume_column == "total_output_satoshis_excl_coinbase":
        drop_columns = ['total_output_satoshis', 'total_fee_satoshis', 'block_number', 'tx_count_check', 'bits']
    
    processed_df = processed_df.drop(columns=drop_columns)

    # Convert satoshis to BTC and rename
    satoshi_columns = ['transaction_volume']
    processed_df[satoshi_columns] = processed_df[satoshi_columns] / 1e8
    
    # Add Efficiency Defined with ratio of n transactions to volume per block
    #processed_df['block_efficiency'] = processed_df['n_transactions'] / processed_df['transaction_volume']
    processed_df['block_efficiency'] = np.where(
    processed_df['transaction_volume'] > 0,
    processed_df['n_transactions'] / processed_df['transaction_volume'],
    np.nan
)

    # Round-robin miner assignment
    processed_df['miner_id'] = processed_df['block_id'] % 100

    # Calculate fee proxy as n_transactions * transaction_volume
    processed_df['fee_proxy'] = processed_df['n_transactions'] * processed_df['transaction_volume']
    

    # Convert the timestamp column from string to proper datetime
    processed_df['timestamp'] = pd.to_datetime(processed_df['timestamp'])
    
    
    # Groupby miner_id
    miner_df = processed_df.groupby('miner_id').agg(
        blocks_mined=('block_id', 'count'),
        avg_transactions=('n_transactions', 'mean'),
        avg_volume=('transaction_volume', 'mean'),
        avg_fee=('fee_proxy', 'mean'),
        fee_volatility=('fee_proxy', 'std'),
        avg_block_size=('size', 'mean'),
        difficulty=('difficulty', 'mean'),
        efficiency=('block_efficiency', 'mean'), # use mean of ratios as default, average of (n transactions / transaction volume per individual block)
        profitability=('fee_proxy', 'sum'),
        last_block_id=('block_id', 'max'),   # last block a mined by a miner
        last_block_time=('timestamp', 'max') # last block mined timestamp
    ).reset_index()
    
    # convert profitability from total sum to avg profitability
    miner_df['profitability'] = miner_df['profitability'] / (miner_df['blocks_mined'] + 1)
    
    if age_definition == "distance_from_tip":
        # Method 1: gap between each miner's last block and the very last block in the whole dataset
        miner_df['age'] = processed_df['timestamp'].max() - miner_df['last_block_time']
        miner_df['age'] = miner_df['age'].dt.total_seconds()
    elif age_definition == "neighbor_gap":
        # Method 2 (Ethan's): sort miners by last-mined time, then gap to the miner right before you
        sorted_miners = miner_df.sort_values('last_block_time').reset_index(drop=True)
        sorted_miners['age'] = sorted_miners['last_block_time'].diff().dt.total_seconds()
        sorted_miners['age'] = sorted_miners['age'].fillna(sorted_miners['age'].median())
        miner_df = miner_df.drop(columns=['age'], errors='ignore').merge(
            sorted_miners[['miner_id', 'age']], on='miner_id'
        )
    
    # Calculate Efficiency Based on Definition of Efficiency
    if efficiency_definition == "ratio_of_means":
        # added very small number (1 * 10^-9) to prevent division by zero
        # if any miner has avg_volume of 0
        miner_df['efficiency'] = miner_df['avg_transactions'] / (miner_df['avg_volume'] + 1e-9)
    elif efficiency_definition == "mean_of_ratios":
        # use mean of ratios as default, average of (n transactions / transaction volume per individual block)
        pass
    
    
    
    # Calculate median efficiency
    median_efficiency = miner_df['efficiency'].median()
    # print(f"Median Efficiency = {median_efficiency}")
    # label miner as 1 if its efficiency is more than median, else 0
    miner_df['label'] = (miner_df['efficiency'] > median_efficiency).astype(int)
    
    return miner_df

In [8]:
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.neural_network import MLPClassifier

seed = 42
miner_df = preprocess(df, "total_output_satoshis", "ratio_of_means", "neighbor_gap")
y = miner_df['label']

feature_sets = {
    "full": ['blocks_mined', 'avg_transactions', 'avg_volume',
             'avg_fee', 'fee_volatility', 'avg_block_size',
             'difficulty', 'profitability', 'age'],
    "reduced": ['blocks_mined',
                'avg_fee', 'fee_volatility', 'avg_block_size',
                'difficulty', 'profitability', 'age'],
}

es_configs = {
    "no_early_stopping": dict(),
    "early_stopping": dict(early_stopping=True, validation_fraction=0.1,
                            n_iter_no_change=20, max_iter=100, batch_size=16),
}

cv = StratifiedKFold(n_splits=5, shuffle=False)

for set_name, cols in feature_sets.items():
    X = miner_df[cols]
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=seed, stratify=y)

    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)

    for es_name, es_kwargs in es_configs.items():
        model = MLPClassifier(
            hidden_layer_sizes=(64, 32, 16, 8), activation='relu', solver='adam',
            alpha=0.001, random_state=seed, **es_kwargs
        )
        model.fit(X_train_scaled, y_train)
        test_acc = model.score(X_test_scaled, y_test)

        cv_scores = cross_val_score(
            MLPClassifier(hidden_layer_sizes=(64, 32, 16, 8), activation='relu',
                          solver='adam', alpha=0.001, random_state=seed, **es_kwargs),
            X_train_scaled, y_train, cv=cv
        )

        print(f"{set_name:8s} | {es_name:18s} | test_acc={test_acc:.4f} | "
              f"cv_mean={cv_scores.mean():.4f} | cv_scores={cv_scores.round(4).tolist()}")

full     | no_early_stopping  | test_acc=0.8000 | cv_mean=0.9250 | cv_scores=[0.875, 0.875, 0.9375, 0.9375, 1.0]
full     | early_stopping     | test_acc=0.9000 | cv_mean=0.7750 | cv_scores=[0.625, 0.625, 0.875, 0.875, 0.875]


d:\Proof of AI Research\poai_simulation\.venv\Lib\site-packages\sklearn\neural_network\_multilayer_perceptron.py:785: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (200) reached and the optimization hasn't converged yet.
  warnings.warn(
d:\Proof of AI Research\poai_simulation\.venv\Lib\site-packages\sklearn\neural_network\_multilayer_perceptron.py:785: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (200) reached and the optimization hasn't converged yet.
  warnings.warn(
d:\Proof of AI Research\poai_simulation\.venv\Lib\site-packages\sklearn\neural_network\_multilayer_perceptron.py:785: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (200) reached and the optimization hasn't converged yet.
  warnings.warn(
d:\Proof of AI Research\poai_simulation\.venv\Lib\site-packages\sklearn\neural_network\_multilayer_perceptron.py:785: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (200) reached and the optimization hasn't converged yet.
  war

reduced  | no_early_stopping  | test_acc=0.7500 | cv_mean=0.7500 | cv_scores=[0.8125, 0.625, 0.6875, 0.6875, 0.9375]
reduced  | early_stopping     | test_acc=0.8500 | cv_mean=0.7750 | cv_scores=[0.8125, 0.625, 0.8125, 0.75, 0.875]


In [9]:
miner_df = preprocess(df, "total_output_satoshis", "mean_of_ratios", "neighbor_gap")
miner_df.sort_values(by='last_block_time', ascending=False).head(10)

,miner_id,blocks_mined,avg_transactions,avg_volume,avg_fee,fee_volatility,avg_block_size,difficulty,efficiency,profitability,last_block_id,last_block_time,age,label
8,8,8110,1111.768187,10136.224152,1.780766e+07,5.113957e+07,635394.452898,7.823222e+12,0.287507,1.780547e+07,810908,2023-10-06 13:37:21+00:00,21.0,0
7,7,8110,1116.194945,10701.430329,1.872428e+07,6.977861e+07,636561.353514,7.823309e+12,0.422076,1.872197e+07,810907,2023-10-06 13:37:00+00:00,2805.0,1
6,6,8110,1117.373490,9933.392749,1.811877e+07,5.012387e+07,636283.305795,7.823309e+12,0.290385,1.811654e+07,810906,2023-10-06 12:50:15+00:00,266.0,0
5,5,8110,1117.982244,10802.388056,1.963532e+07,7.151489e+07,636626.161652,7.823309e+12,0.284028,1.963290e+07,810905,2023-10-06 12:45:49+00:00,544.0,0
4,4,8110,1123.225524,11427.595817,2.072525e+07,1.492980e+08,633191.847965,7.823309e+12,0.411655,2.072270e+07,810904,2023-10-06 12:36:45+00:00,544.0,1
3,3,8110,1099.984957,10317.032438,1.838393e+07,5.959257e+07,631024.890999,7.822986e+12,0.366910,1.838166e+07,810903,2023-10-06 12:27:41+00:00,759.0,1
2,2,8110,1120.462762,10749.633443,1.934361e+07,9.517518e+07,632985.547472,7.822986e+12,0.386658,1.934123e+07,810902,2023-10-06 12:15:02+00:00,837.0,1
1,1,8110,1113.248212,10876.172794,1.882207e+07,5.546725e+07,633592.245993,7.822986e+12,0.310413,1.881975e+07,810901,2023-10-06 12:01:05+00:00,386.0,0
0,0,8110,1112.876079,10848.979162,1.910743e+07,7.302996e+07,636997.443157,7.822986e+12,0.351540,1.910507e+07,810900,2023-10-06 11:54:39+00:00,30.0,1
99,99,8109,1123.059440,11095.425626,1.924070e+07,6.450264e+07,639058.787520,7.824585e+12,0.351457,1.923833e+07,810899,2023-10-06 11:54:09+00:00,8.0,1


In [10]:
# Train and Test
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.neural_network import MLPClassifier
from sklearn.model_selection import cross_val_score
from sklearn.metrics import accuracy_score

def train_and_test(miner_df, seed, feature_set):
    
    feature_columns = []
    if feature_set == "full":
        feature_columns = ['blocks_mined', 'avg_transactions', 'avg_volume', 
                        'avg_fee', 'fee_volatility', 'avg_block_size', 
                        'difficulty', 'profitability', 'age']
    elif feature_set == "reduced":
        feature_columns = ['blocks_mined', 
                           'avg_fee', 'fee_volatility', 'avg_block_size', 
                           'difficulty', 'profitability', 'age']

    X = miner_df[feature_columns]
    y = miner_df['label']
    
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=seed, stratify=y)

    # Scale data using normalization so that different features
    # with huge numbers and small numbers have equal importance

    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)
    
    model = MLPClassifier(
        hidden_layer_sizes=(64, 32, 16, 8), 
        activation='relu', 
        solver='adam', 
        alpha=0.001, 
        random_state=seed,
        early_stopping=True,
        validation_fraction=0.1,
        n_iter_no_change=20,
        max_iter=100,
        batch_size=16
        )

    # print(model)

    # Fit the model (Training)
    model.fit(X_train_scaled, y_train)
    
    # Test the accuracy of the model
    # Ethan's accuracy:
    # Test Accuracy = 95%
    # 5-fold Cross Validation Accuracy = 48.75%
    y_prediction = model.predict(X_test_scaled)
    # print(f"y prediction: {y_prediction}")
    
    test_accuracy = accuracy_score(y_test, y_prediction)
    # print(test_accuracy)

    cv_scores = cross_val_score(model, X_train_scaled, y_train, cv=5)
    cv_mean = cv_scores.mean()
    # print(f"Cross-Validation Scores:     {cv_scores}")
    # print(f"Cross-Validation Mean Score: {cv_mean}")
    
    median_efficiency = miner_df['efficiency'].median()
    min_efficiency = miner_df['efficiency'].min()
    max_efficiency = miner_df['efficiency'].max()
    
    
    # High cv score found? should be closer to Ethan's Accuracy
    # corr_avg_volume_efficiency = miner_df[['avg_transactions', 'avg_volume']].corrwith(miner_df['efficiency'])
    corr_avg_transaction_efficiency = miner_df['avg_transactions'].corr(miner_df['efficiency'])
    corr_avg_volume_efficiency = miner_df['avg_volume'].corr(miner_df['efficiency'])
    
    # print(corr_avg_transaction_efficiency)
    # print(corr_avg_volume_efficiency)
    
    return dict(
        test_accuracy=test_accuracy, cv_mean=cv_mean, 
        median_efficiency=median_efficiency, min_efficiency=min_efficiency, max_efficiency=max_efficiency,
        corr_avg_transaction_efficiency=corr_avg_transaction_efficiency, corr_avg_volume_efficiency=corr_avg_volume_efficiency,
        model_iterations=model.n_iter_
        )

In [11]:
# total_output_satoshis, full features
train_and_test(miner_df, 42, "full")

{'test_accuracy': 0.45,
 'cv_mean': np.float64(0.4375),
 'median_efficiency': np.float64(0.3181217328568209),
 'min_efficiency': np.float64(0.2735800481872434),
 'max_efficiency': np.float64(0.45054328283273865),
 'corr_avg_transaction_efficiency': np.float64(0.11250176535326101),
 'corr_avg_volume_efficiency': np.float64(0.03423481197431949),
 'model_iterations': 29}

In [12]:
def preprocess_and_test(df, seed, volume_column, efficiency_definition, age_definition, feature_set):
    miner_df = preprocess(df, volume_column, efficiency_definition, age_definition)
    results = train_and_test(miner_df, seed, feature_set)
    full_results = {
        "seed": seed,
        "feature_set": feature_set,
        "volume_column": volume_column,
        **results
    }
    
    return pd.DataFrame([full_results])

In [13]:
results_df = []

# Ratio of Means
results_df.append(preprocess_and_test(df, 42, "total_output_satoshis_excl_coinbase", "ratio_of_means", "neighbor_gap", "full"))
results_df.append(preprocess_and_test(df, 42, "total_output_satoshis_excl_coinbase", "ratio_of_means", "neighbor_gap", "reduced"))
results_df.append(preprocess_and_test(df, 42, "total_output_satoshis", "ratio_of_means", "neighbor_gap", "full"))
results_df.append(preprocess_and_test(df, 42, "total_output_satoshis", "ratio_of_means", "neighbor_gap", "reduced"))

# Mean of Ratios
results_df.append(preprocess_and_test(df, 42, "total_output_satoshis_excl_coinbase", "mean_of_ratios", "neighbor_gap", "full"))
results_df.append(preprocess_and_test(df, 42, "total_output_satoshis_excl_coinbase", "mean_of_ratios", "neighbor_gap", "reduced"))
results_df.append(preprocess_and_test(df, 42, "total_output_satoshis", "mean_of_ratios", "neighbor_gap", "full"))
results_df.append(preprocess_and_test(df, 42, "total_output_satoshis", "mean_of_ratios", "neighbor_gap", "reduced"))

final_results = pd.concat(results_df, ignore_index=True)
final_results

,seed,feature_set,volume_column,test_accuracy,cv_mean,median_efficiency,min_efficiency,max_efficiency,corr_avg_transaction_efficiency,corr_avg_volume_efficiency,model_iterations
0,42,full,total_output_satoshis_excl_coinbase,0.90,0.7875,0.107405,0.098501,0.114919,-0.000700,-0.982983,25
1,42,reduced,total_output_satoshis_excl_coinbase,0.85,0.7750,0.107405,0.098501,0.114919,-0.000700,-0.982983,37
2,42,full,total_output_satoshis,0.90,0.7750,0.107153,0.098291,0.114630,-0.000307,-0.982917,25
3,42,reduced,total_output_satoshis,0.85,0.7750,0.107153,0.098291,0.114630,-0.000307,-0.982917,37
4,42,full,total_output_satoshis_excl_coinbase,0.60,0.5000,0.575491,0.417039,27766.783956,0.062137,0.094429,56
5,42,reduced,total_output_satoshis_excl_coinbase,0.50,0.4875,0.575491,0.417039,27766.783956,0.062137,0.094429,22
6,42,full,total_output_satoshis,0.45,0.4375,0.318122,0.273580,0.450543,0.112502,0.034235,29
7,42,reduced,total_output_satoshis,0.55,0.5000,0.318122,0.273580,0.450543,0.112502,0.034235,40


In [14]:
final_results.groupby(['volume_column', 'feature_set'])['cv_mean'].agg(['mean', 'std'])

mean       std
volume_column                       feature_set                   
total_output_satoshis               full         0.60625  0.238649
                                    reduced      0.63750  0.194454
total_output_satoshis_excl_coinbase full         0.64375  0.203293
                                    reduced      0.63125  0.203293

In [15]:
# Ratio of Means
seeds = [1, 3, 15, 17, 25, 29, 30, 36, 42, 50, 51, 67, 100]
diff_seed_results_df_RoM = []
for seed in seeds:
    diff_seed_results_df_RoM.append(preprocess_and_test(df, seed, "total_output_satoshis_excl_coinbase", "ratio_of_means", "neighbor_gap", "full"))
    diff_seed_results_df_RoM.append(preprocess_and_test(df, seed, "total_output_satoshis_excl_coinbase", "ratio_of_means", "neighbor_gap", "reduced"))
    diff_seed_results_df_RoM.append(preprocess_and_test(df, seed, "total_output_satoshis", "ratio_of_means", "neighbor_gap", "full"))
    diff_seed_results_df_RoM.append(preprocess_and_test(df, seed, "total_output_satoshis", "ratio_of_means", "neighbor_gap", "reduced"))

diff_seed_final_results_RoM = pd.concat(diff_seed_results_df_RoM, ignore_index=True)
diff_seed_final_results_RoM

,seed,feature_set,volume_column,test_accuracy,cv_mean,median_efficiency,min_efficiency,max_efficiency,corr_avg_transaction_efficiency,corr_avg_volume_efficiency,model_iterations
0,1,full,total_output_satoshis_excl_coinbase,0.75,0.8000,0.107405,0.098501,0.114919,-0.000700,-0.982983,27
1,1,reduced,total_output_satoshis_excl_coinbase,0.80,0.6875,0.107405,0.098501,0.114919,-0.000700,-0.982983,63
2,1,full,total_output_satoshis,0.75,0.8000,0.107153,0.098291,0.114630,-0.000307,-0.982917,27
3,1,reduced,total_output_satoshis,0.80,0.6875,0.107153,0.098291,0.114630,-0.000307,-0.982917,63
4,3,full,total_output_satoshis_excl_coinbase,0.85,0.8375,0.107405,0.098501,0.114919,-0.000700,-0.982983,38
5,3,reduced,total_output_satoshis_excl_coinbase,0.75,0.6750,0.107405,0.098501,0.114919,-0.000700,-0.982983,29
6,3,full,total_output_satoshis,0.85,0.8375,0.107153,0.098291,0.114630,-0.000307,-0.982917,38
7,3,reduced,total_output_satoshis,0.75,0.6750,0.107153,0.098291,0.114630,-0.000307,-0.982917,29
8,15,full,total_output_satoshis_excl_coinbase,0.80,0.8000,0.107405,0.098501,0.114919,-0.000700,-0.982983,41
9,15,reduced,total_output_satoshis_excl_coinbase,0.85,0.7500,0.107405,0.098501,0.114919,-0.000700,-0.982983,28


In [16]:
diff_seed_final_results_RoM.groupby(['volume_column', 'feature_set'])['cv_mean'].agg(['mean', 'std'])

mean       std
volume_column                       feature_set                    
total_output_satoshis               full         0.783654  0.067001
                                    reduced      0.709615  0.054761
total_output_satoshis_excl_coinbase full         0.781731  0.070824
                                    reduced      0.708654  0.055758

In [17]:
# Mean of Ratios
seeds = [1, 3, 15, 17, 25, 29, 30, 36, 42, 50, 51, 67, 100]
diff_seed_results_df_MoR = []
for seed in seeds:
    diff_seed_results_df_MoR.append(preprocess_and_test(df, seed, "total_output_satoshis_excl_coinbase", "mean_of_ratios", "neighbor_gap", "full"))
    diff_seed_results_df_MoR.append(preprocess_and_test(df, seed, "total_output_satoshis_excl_coinbase", "mean_of_ratios", "neighbor_gap", "reduced"))
    diff_seed_results_df_MoR.append(preprocess_and_test(df, seed, "total_output_satoshis", "mean_of_ratios", "neighbor_gap", "full"))
    diff_seed_results_df_MoR.append(preprocess_and_test(df, seed, "total_output_satoshis", "mean_of_ratios", "neighbor_gap", "reduced"))

diff_seed_final_results_MoR = pd.concat(diff_seed_results_df_MoR, ignore_index=True)
diff_seed_final_results_MoR

,seed,feature_set,volume_column,test_accuracy,cv_mean,median_efficiency,min_efficiency,max_efficiency,corr_avg_transaction_efficiency,corr_avg_volume_efficiency,model_iterations
0,1,full,total_output_satoshis_excl_coinbase,0.40,0.5250,0.575491,0.417039,27766.783956,0.062137,0.094429,35
1,1,reduced,total_output_satoshis_excl_coinbase,0.50,0.5000,0.575491,0.417039,27766.783956,0.062137,0.094429,43
2,1,full,total_output_satoshis,0.50,0.4625,0.318122,0.273580,0.450543,0.112502,0.034235,31
3,1,reduced,total_output_satoshis,0.50,0.4875,0.318122,0.273580,0.450543,0.112502,0.034235,22
4,3,full,total_output_satoshis_excl_coinbase,0.50,0.5125,0.575491,0.417039,27766.783956,0.062137,0.094429,22
5,3,reduced,total_output_satoshis_excl_coinbase,0.60,0.5500,0.575491,0.417039,27766.783956,0.062137,0.094429,33
6,3,full,total_output_satoshis,0.50,0.5000,0.318122,0.273580,0.450543,0.112502,0.034235,22
7,3,reduced,total_output_satoshis,0.55,0.4375,0.318122,0.273580,0.450543,0.112502,0.034235,50
8,15,full,total_output_satoshis_excl_coinbase,0.50,0.5125,0.575491,0.417039,27766.783956,0.062137,0.094429,22
9,15,reduced,total_output_satoshis_excl_coinbase,0.60,0.5375,0.575491,0.417039,27766.783956,0.062137,0.094429,26


In [18]:
diff_seed_final_results_MoR.groupby(['volume_column', 'feature_set'])['cv_mean'].agg(['mean', 'std'])

mean       std
volume_column                       feature_set                    
total_output_satoshis               full         0.463462  0.059613
                                    reduced      0.474038  0.036962
total_output_satoshis_excl_coinbase full         0.505769  0.026327
                                    reduced      0.513462  0.026742

# Permutation Importance

## Ratio of Means

In [19]:
# --- Shared setup (same for both feature sets) ---
seed = 42
miner_df = preprocess(df, "total_output_satoshis", "ratio_of_means", "neighbor_gap")
y = miner_df['label']

# --- Full Feature Set ---
feature_columns_full = ['blocks_mined', 'avg_transactions', 'avg_volume',
                         'avg_fee', 'fee_volatility', 'avg_block_size',
                         'difficulty', 'profitability', 'age']
X_full = miner_df[feature_columns_full]
X_train_full, X_test_full, y_train_full, y_test_full = train_test_split(
    X_full, y, test_size=0.2, random_state=seed, stratify=y)

scaler_full = StandardScaler()
X_train_scaled_full = scaler_full.fit_transform(X_train_full)
X_test_scaled_full = scaler_full.transform(X_test_full)

model_full = MLPClassifier(
    hidden_layer_sizes=(64, 32, 16, 8), activation='relu', solver='adam', alpha=0.001,
    random_state=seed, early_stopping=True, validation_fraction=0.1,
    n_iter_no_change=20, max_iter=100, batch_size=16
)
model_full.fit(X_train_scaled_full, y_train_full)

# --- Reduced Feature Set ---
feature_columns_reduced = ['blocks_mined',
                            'avg_fee', 'fee_volatility', 'avg_block_size',
                            'difficulty', 'profitability', 'age']
X_reduced = miner_df[feature_columns_reduced]
X_train_reduced, X_test_reduced, y_train_reduced, y_test_reduced = train_test_split(
    X_reduced, y, test_size=0.2, random_state=seed, stratify=y)

scaler_reduced = StandardScaler()
X_train_scaled_reduced = scaler_reduced.fit_transform(X_train_reduced)
X_test_scaled_reduced = scaler_reduced.transform(X_test_reduced)

model_reduced = MLPClassifier(
    hidden_layer_sizes=(64, 32, 16, 8), activation='relu', solver='adam', alpha=0.001,
    random_state=seed, early_stopping=True, validation_fraction=0.1,
    n_iter_no_change=20, max_iter=100, batch_size=16
)
model_reduced.fit(X_train_scaled_reduced, y_train_reduced)

,"hidden_layer_sizes hidden_layer_sizes: array-like of shape(n_layers - 2,), default=(100,)The ith element represents the number of neurons in the ithhidden layer.","(64, ...)"
,"alpha alpha: float, default=0.0001Strength of the L2 regularization term. The L2 regularization termis divided by the sample size when added to the loss.For an example usage and visualization of varying regularization, see:ref:`sphx_glr_auto_examples_neural_networks_plot_mlp_alpha.py`.",0.001
,"batch_size batch_size: int, default='auto'Size of minibatches for stochastic optimizers.If the solver is 'lbfgs', the classifier will not use minibatch.When set to ""auto"", `batch_size=min(200, n_samples)`.",16
,"max_iter max_iter: int, default=200Maximum number of iterations. The solver iterates until convergence(determined by 'tol') or this number of iterations. For stochasticsolvers ('sgd', 'adam'), note that this determines the number of epochs(how many times each data point will be used), not the number ofgradient steps.",100
,"random_state random_state: int, RandomState instance, default=NoneDetermines random number generation for weights and biasinitialization, train-test split if early stopping is used, and batchsampling when solver='sgd' or 'adam'.Pass an int for reproducible results across multiple function calls.See :term:`Glossary <random_state>`.",42
,"early_stopping early_stopping: bool, default=FalseWhether to use early stopping to terminate training when validationscore is not improving. If set to True, it will automatically setaside ``validation_fraction`` of training data as validation andterminate training when validation score is not improving by at least``tol`` for ``n_iter_no_change`` consecutive epochs. The split isstratified, except in a multilabel setting.If early stopping is False, then the training stops when the trainingloss does not improve by more than ``tol`` for ``n_iter_no_change``consecutive passes over the training set.Only effective when solver='sgd' or 'adam'.",True
,"n_iter_no_change n_iter_no_change: int, default=10Maximum number of epochs to not meet ``tol`` improvement.Only effective when solver='sgd' or 'adam'... versionadded:: 0.20",20
,"activation activation: {'identity', 'logistic', 'tanh', 'relu'}, default='relu'Activation function for the hidden layer.- 'identity', no-op activation, useful to implement linear bottleneck, returns f(x) = x- 'logistic', the logistic sigmoid function, returns f(x) = 1 / (1 + exp(-x)).- 'tanh', the hyperbolic tan function, returns f(x) = tanh(x).- 'relu', the rectified linear unit function, returns f(x) = max(0, x)",'relu'
,"solver solver: {'lbfgs', 'sgd', 'adam'}, default='adam'The solver for weight optimization.- 'lbfgs' is an optimizer in the family of quasi-Newton methods.- 'sgd' refers to stochastic gradient descent.- 'adam' refers to a stochastic gradient-based optimizer proposed by Kingma, Diederik, and Jimmy BaFor a comparison between Adam optimizer and SGD, see:ref:`sphx_glr_auto_examples_neural_networks_plot_mlp_training_curves.py`.Note: The default solver 'adam' works pretty well on relativelylarge datasets (with thousands of training samples or more) in terms ofboth training time and validation score.For small datasets, however, 'lbfgs' can converge faster and performbetter.",'adam'
,"learning_rate learning_rate: {'constant', 'invscaling', 'adaptive'}, default='constant'Learning rate schedule for weight updates.- 'constant' is a constant learning rate given by 'learning_rate_init'.- 'invscaling' gradually decreases the learning rate at each time step 't' using an inverse scaling exponent of 'power_t'. effective_learning_rate = learning_rate_init / pow(t, power_t)- 'adaptive' keeps the learning rate constant to 'learning_rate_init' as long as training loss keeps decreasing. Each time two consecutive epochs fail to decrease training loss by at least tol, or fail to increase validation score by at least tol if 'early_stopping' is on, the current learning rate is divided by 5.O

In [20]:
from sklearn.inspection import permutation_importance

result_full = permutation_importance(
    model_full, X_test_scaled_full, y_test_full,
    scoring='accuracy', n_repeats=30, random_state=42
)

result_reduced = permutation_importance(
    model_reduced, X_test_scaled_reduced, y_test_reduced,
    scoring='accuracy', n_repeats=30, random_state=42
)

In [21]:
importance_full_df = pd.DataFrame({
    'feature': feature_columns_full,
    'importance_mean': result_full.importances_mean,
    'importance_std': result_full.importances_std
}).sort_values('importance_mean', ascending=False)

importance_reduced_df = pd.DataFrame({
    'feature': feature_columns_reduced,
    'importance_mean': result_reduced.importances_mean,
    'importance_std': result_reduced.importances_std
}).sort_values('importance_mean', ascending=False)

print(importance_full_df)
print(importance_reduced_df)

            feature  importance_mean  importance_std
7     profitability         0.176667        0.092856
6        difficulty         0.158333        0.053359
3           avg_fee         0.141667        0.076467
2        avg_volume         0.136667        0.051532
4    fee_volatility         0.050000        0.046547
1  avg_transactions         0.048333        0.050799
0      blocks_mined         0.011667        0.030777
8               age         0.008333        0.046696
5    avg_block_size         0.006667        0.040277
          feature  importance_mean  importance_std
1         avg_fee         0.200000        0.083666
5   profitability         0.126667        0.057349
3  avg_block_size         0.126667        0.051208
4      difficulty         0.060000        0.045461
2  fee_volatility         0.056667        0.040277
6             age         0.018333        0.027335
0    blocks_mined        -0.033333        0.023570


## Mean of Ratios

In [22]:
# --- Shared setup (same for both feature sets) ---
seed = 42
miner_df = preprocess(df, "total_output_satoshis", "mean_of_ratios", "neighbor_gap")
y = miner_df['label']

# --- Full Feature Set ---
feature_columns_full = ['blocks_mined', 'avg_transactions', 'avg_volume',
                         'avg_fee', 'fee_volatility', 'avg_block_size',
                         'difficulty', 'profitability', 'age']
X_full = miner_df[feature_columns_full]
X_train_full, X_test_full, y_train_full, y_test_full = train_test_split(
    X_full, y, test_size=0.2, random_state=seed, stratify=y)

scaler_full = StandardScaler()
X_train_scaled_full = scaler_full.fit_transform(X_train_full)
X_test_scaled_full = scaler_full.transform(X_test_full)

model_full = MLPClassifier(
    hidden_layer_sizes=(64, 32, 16, 8), activation='relu', solver='adam', alpha=0.001,
    random_state=seed, early_stopping=True, validation_fraction=0.1,
    n_iter_no_change=20, max_iter=100, batch_size=16
)
model_full.fit(X_train_scaled_full, y_train_full)

# --- Reduced Feature Set ---
feature_columns_reduced = ['blocks_mined',
                            'avg_fee', 'fee_volatility', 'avg_block_size',
                            'difficulty', 'profitability', 'age']
X_reduced = miner_df[feature_columns_reduced]
X_train_reduced, X_test_reduced, y_train_reduced, y_test_reduced = train_test_split(
    X_reduced, y, test_size=0.2, random_state=seed, stratify=y)

scaler_reduced = StandardScaler()
X_train_scaled_reduced = scaler_reduced.fit_transform(X_train_reduced)
X_test_scaled_reduced = scaler_reduced.transform(X_test_reduced)

model_reduced = MLPClassifier(
    hidden_layer_sizes=(64, 32, 16, 8), activation='relu', solver='adam', alpha=0.001,
    random_state=seed, early_stopping=True, validation_fraction=0.1,
    n_iter_no_change=20, max_iter=100, batch_size=16
)
model_reduced.fit(X_train_scaled_reduced, y_train_reduced)

,"hidden_layer_sizes hidden_layer_sizes: array-like of shape(n_layers - 2,), default=(100,)The ith element represents the number of neurons in the ithhidden layer.","(64, ...)"
,"alpha alpha: float, default=0.0001Strength of the L2 regularization term. The L2 regularization termis divided by the sample size when added to the loss.For an example usage and visualization of varying regularization, see:ref:`sphx_glr_auto_examples_neural_networks_plot_mlp_alpha.py`.",0.001
,"batch_size batch_size: int, default='auto'Size of minibatches for stochastic optimizers.If the solver is 'lbfgs', the classifier will not use minibatch.When set to ""auto"", `batch_size=min(200, n_samples)`.",16
,"max_iter max_iter: int, default=200Maximum number of iterations. The solver iterates until convergence(determined by 'tol') or this number of iterations. For stochasticsolvers ('sgd', 'adam'), note that this determines the number of epochs(how many times each data point will be used), not the number ofgradient steps.",100
,"random_state random_state: int, RandomState instance, default=NoneDetermines random number generation for weights and biasinitialization, train-test split if early stopping is used, and batchsampling when solver='sgd' or 'adam'.Pass an int for reproducible results across multiple function calls.See :term:`Glossary <random_state>`.",42
,"early_stopping early_stopping: bool, default=FalseWhether to use early stopping to terminate training when validationscore is not improving. If set to True, it will automatically setaside ``validation_fraction`` of training data as validation andterminate training when validation score is not improving by at least``tol`` for ``n_iter_no_change`` consecutive epochs. The split isstratified, except in a multilabel setting.If early stopping is False, then the training stops when the trainingloss does not improve by more than ``tol`` for ``n_iter_no_change``consecutive passes over the training set.Only effective when solver='sgd' or 'adam'.",True
,"n_iter_no_change n_iter_no_change: int, default=10Maximum number of epochs to not meet ``tol`` improvement.Only effective when solver='sgd' or 'adam'... versionadded:: 0.20",20
,"activation activation: {'identity', 'logistic', 'tanh', 'relu'}, default='relu'Activation function for the hidden layer.- 'identity', no-op activation, useful to implement linear bottleneck, returns f(x) = x- 'logistic', the logistic sigmoid function, returns f(x) = 1 / (1 + exp(-x)).- 'tanh', the hyperbolic tan function, returns f(x) = tanh(x).- 'relu', the rectified linear unit function, returns f(x) = max(0, x)",'relu'
,"solver solver: {'lbfgs', 'sgd', 'adam'}, default='adam'The solver for weight optimization.- 'lbfgs' is an optimizer in the family of quasi-Newton methods.- 'sgd' refers to stochastic gradient descent.- 'adam' refers to a stochastic gradient-based optimizer proposed by Kingma, Diederik, and Jimmy BaFor a comparison between Adam optimizer and SGD, see:ref:`sphx_glr_auto_examples_neural_networks_plot_mlp_training_curves.py`.Note: The default solver 'adam' works pretty well on relativelylarge datasets (with thousands of training samples or more) in terms ofboth training time and validation score.For small datasets, however, 'lbfgs' can converge faster and performbetter.",'adam'
,"learning_rate learning_rate: {'constant', 'invscaling', 'adaptive'}, default='constant'Learning rate schedule for weight updates.- 'constant' is a constant learning rate given by 'learning_rate_init'.- 'invscaling' gradually decreases the learning rate at each time step 't' using an inverse scaling exponent of 'power_t'. effective_learning_rate = learning_rate_init / pow(t, power_t)- 'adaptive' keeps the learning rate constant to 'learning_rate_init' as long as training loss keeps decreasing. Each time two consecutive epochs fail to decrease training loss by at least tol, or fail to increase validation score by at least tol if 'early_stopping' is on, the current learning rate is divided by 5.O

In [23]:
result_full = permutation_importance(
    model_full, X_test_scaled_full, y_test_full,
    scoring='accuracy', n_repeats=30, random_state=42
)

result_reduced = permutation_importance(
    model_reduced, X_test_scaled_reduced, y_test_reduced,
    scoring='accuracy', n_repeats=30, random_state=42
)

In [24]:
importance_full_df = pd.DataFrame({
    'feature': feature_columns_full,
    'importance_mean': result_full.importances_mean,
    'importance_std': result_full.importances_std
}).sort_values('importance_mean', ascending=False)

importance_reduced_df = pd.DataFrame({
    'feature': feature_columns_reduced,
    'importance_mean': result_reduced.importances_mean,
    'importance_std': result_reduced.importances_std
}).sort_values('importance_mean', ascending=False)

print(importance_full_df)
print(importance_reduced_df)

            feature  importance_mean  importance_std
4    fee_volatility         0.035000        0.034521
8               age        -0.008333        0.042979
5    avg_block_size        -0.021667        0.069142
2        avg_volume        -0.043333        0.047842
0      blocks_mined        -0.045000        0.045369
3           avg_fee        -0.053333        0.036362
1  avg_transactions        -0.055000        0.085975
6        difficulty        -0.056667        0.086346
7     profitability        -0.070000        0.037859
          feature  importance_mean  importance_std
4      difficulty         0.083333        0.058214
2  fee_volatility         0.040000        0.058310
5   profitability         0.016667        0.056765
1         avg_fee         0.011667        0.067926
0    blocks_mined        -0.008333        0.038909
3  avg_block_size        -0.045000        0.079948
6             age        -0.048333        0.065171


In [25]:
print(f"full model:    {model_full.score(X_test_scaled_full, y_test_full)}")
print(f"reduced model: {model_reduced.score(X_test_scaled_reduced, y_test_reduced)}")

full model:    0.45
reduced model: 0.55


# Kaggle Dataset Test

In [26]:
kaggle_df = pd.read_csv("../../data/kaggle_dataset.csv")

adapted = kaggle_df.rename(columns={
    "height": "number",
    "tx_count": "transaction_count",
    "output_amount": "total_output_satoshis_excl_coinbase",
})

# columns preprocess() expects to exist only so it can drop them — never used downstream
for col in ["total_output_satoshis", "total_fee_satoshis", "block_number", "tx_count_check", "bits"]:
    adapted[col] = 0

miner_df_kaggle = preprocess(adapted, "total_output_satoshis_excl_coinbase", "ratio_of_means", "neighbor_gap")

In [27]:
train_and_test(miner_df_kaggle, 42, "full")

{'test_accuracy': 0.9,
 'cv_mean': np.float64(0.8125),
 'median_efficiency': np.float64(0.10740506578081713),
 'min_efficiency': np.float64(0.09850087910470431),
 'max_efficiency': np.float64(0.1149185034367631),
 'corr_avg_transaction_efficiency': np.float64(-0.001046223204879258),
 'corr_avg_volume_efficiency': np.float64(-0.9829945688110305),
 'model_iterations': 25}

In [28]:
# --- Step 1: Restrict both datasets to the EXACT same shared block range ---
# BigQuery: blocks 0-810908 (has genesis, missing Kaggle's extra tip block)
# Kaggle:   blocks 1-810909 (missing genesis, has one extra tip block)
# Shared range: 1 to 810908 inclusive (810,908 blocks)

df_shared = df[(df['number'] >= 1) & (df['number'] <= 810908)].reset_index(drop=True)
adapted_shared = adapted[(adapted['number'] >= 1) & (adapted['number'] <= 810908)].reset_index(drop=True)

print(f"BigQuery shared rows: {len(df_shared)}")
print(f"Kaggle shared rows:   {len(adapted_shared)}")

# --- Step 2: Verify the raw shared columns actually match beyond total_fees ---
# (we only ever checked total_fees == total_fee_satoshis before — check the others too)
merged_check = df_shared.merge(adapted_shared, on="number", suffixes=("_bq", "_kaggle"))

for col in ["size", "transaction_count", "difficulty"]:
    mismatches = (merged_check[f"{col}_bq"] != merged_check[f"{col}_kaggle"]).sum()
    print(f"{col}: {mismatches} mismatches out of {len(merged_check)}")

# --- Step 3: Run the IDENTICAL pipeline on both, same volume definition ---
# Using excl_coinbase on both sides since that's all Kaggle has (output_amount)
miner_df_bq = preprocess(df_shared, "total_output_satoshis_excl_coinbase", "ratio_of_means", "neighbor_gap")
miner_df_kaggle = preprocess(adapted_shared, "total_output_satoshis_excl_coinbase", "ratio_of_means", "neighbor_gap")

# --- Step 4: Check whether the resulting per-miner feature tables are IDENTICAL ---
feature_cols = ['blocks_mined', 'avg_transactions', 'avg_volume', 'avg_fee',
                'fee_volatility', 'avg_block_size', 'difficulty', 'profitability',
                'age', 'efficiency', 'label']

compare_df = miner_df_bq[['miner_id'] + feature_cols].merge(
    miner_df_kaggle[['miner_id'] + feature_cols], on='miner_id', suffixes=('_bq', '_kaggle')
)

print("\nMax absolute difference per feature (should be ~0 if truly identical):")
for col in feature_cols:
    diff = (compare_df[f"{col}_bq"] - compare_df[f"{col}_kaggle"]).abs().max()
    print(f"  {col}: {diff}")

# --- Step 5: Run the model on both and compare results directly ---
results_bq = train_and_test(miner_df_bq, 42, "full")
results_kaggle = train_and_test(miner_df_kaggle, 42, "full")

print("\nBigQuery (excl_coinbase, shared blocks):", results_bq)
print("Kaggle   (excl_coinbase, shared blocks):", results_kaggle)

BigQuery shared rows: 810908
Kaggle shared rows:   810908
size: 0 mismatches out of 810908
transaction_count: 0 mismatches out of 810908
difficulty: 324439 mismatches out of 810908

Max absolute difference per feature (should be ~0 if truly identical):
  blocks_mined: 0
  avg_transactions: 0.0
  avg_volume: 0.0
  avg_fee: 0.0
  fee_volatility: 0.0
  avg_block_size: 0.0
  difficulty: 0.0009765625
  profitability: 0.0
  age: 2804.997195
  efficiency: 0.0
  label: 0

BigQuery (excl_coinbase, shared blocks): {'test_accuracy': 0.9, 'cv_mean': np.float64(0.8125), 'median_efficiency': np.float64(0.10740506578081713), 'min_efficiency': np.float64(0.09850087910470431), 'max_efficiency': np.float64(0.1149185034367631), 'corr_avg_transaction_efficiency': np.float64(-0.0009654908071716568), 'corr_avg_volume_efficiency': np.float64(-0.9829851669558131), 'model_iterations': 25}
Kaggle   (excl_coinbase, shared blocks): {'test_accuracy': 0.9, 'cv_mean': np.float64(0.8125), 'median_efficiency': np.floa

In [29]:
# --- Rebuild the Kaggle adapter, this time keeping total_fees for testing ---
kaggle_df = pd.read_csv("../../data/kaggle_dataset.csv")

adapted = kaggle_df.rename(columns={
    "height": "number",
    "tx_count": "transaction_count",
    "output_amount": "total_output_satoshis_excl_coinbase",
    "total_fees": "total_fee_satoshis",  # keep the REAL column this time
})

for col in ["total_output_satoshis", "block_number", "tx_count_check", "bits"]:
    adapted[col] = 0

adapted_shared = adapted[(adapted['number'] >= 1) & (adapted['number'] <= 810908)].reset_index(drop=True)

# --- Compute the real-fee-based profitability BEFORE calling preprocess() ---
# (preprocess() drops total_fee_satoshis internally, so this has to happen outside it)
adapted_shared['miner_id'] = adapted_shared['number'] % 100
real_fee_per_miner = adapted_shared.groupby('miner_id')['total_fee_satoshis'].sum().rename('total_real_fees')

# --- Run the normal (synthetic-proxy) pipeline as before ---
miner_df_proxy = preprocess(adapted_shared, "total_output_satoshis_excl_coinbase", "ratio_of_means", "neighbor_gap")
miner_df_test = miner_df_proxy.merge(real_fee_per_miner, on='miner_id')
miner_df_test['profitability_real'] = miner_df_test['total_real_fees'] / (miner_df_test['blocks_mined'] + 1)

# --- Does the REAL version correlate with avg_fee the way the proxy does? ---
corr_proxy = miner_df_test['profitability'].corr(miner_df_test['avg_fee'])
corr_real = miner_df_test['profitability_real'].corr(miner_df_test['avg_fee'])
print(f"proxy profitability  vs avg_fee: {corr_proxy:.4f}  (Ethan reports 1.00)")
print(f"REAL total_fees prof. vs avg_fee: {corr_real:.4f}")

# --- Swap profitability -> profitability_real and re-run the model ---
miner_df_swapped = miner_df_test.copy()
miner_df_swapped['profitability'] = miner_df_swapped['profitability_real']

results_real_fee = train_and_test(miner_df_swapped, 42, "full")
print("\nWith REAL total_fees as profitability:", results_real_fee)
print("(compare against the proxy-based run: test_accuracy=0.9, cv_mean=0.8125)")

proxy profitability  vs avg_fee: 1.0000  (Ethan reports 1.00)
REAL total_fees prof. vs avg_fee: 0.0189

With REAL total_fees as profitability: {'test_accuracy': 0.8, 'cv_mean': np.float64(0.7625), 'median_efficiency': np.float64(0.10740506578081713), 'min_efficiency': np.float64(0.09850087910470431), 'max_efficiency': np.float64(0.1149185034367631), 'corr_avg_transaction_efficiency': np.float64(-0.0009654908071716568), 'corr_avg_volume_efficiency': np.float64(-0.9829851669558131), 'model_iterations': 27}
(compare against the proxy-based run: test_accuracy=0.9, cv_mean=0.8125)


In [30]:
# --- Build per-miner real fee-rate aggregates (fields Ethan never had) ---
kaggle_shared = kaggle_df[(kaggle_df['height'] >= 1) & (kaggle_df['height'] <= 810908)].reset_index(drop=True)
kaggle_shared['miner_id'] = kaggle_shared['height'] % 100

fee_rate_features = kaggle_shared.groupby('miner_id').agg(
    avg_fee_rate_real=('avg_fee_rate', 'mean'),
    median_fee_rate_real=('median_fee_rate', 'mean'),
    fee_range_min_real=('fee_range_min', 'mean'),
    fee_range_max_real=('fee_range_max', 'mean'),
    avg_input_count=('input_count', 'mean'),
    avg_output_count=('output_count', 'mean'),
).reset_index()

miner_df_extended = miner_df_proxy.merge(fee_rate_features, on='miner_id')

# --- Candidate: efficiency defined from a REAL, independent signal instead of the tx_count/volume ratio ---
miner_df_extended['efficiency_real_fee'] = miner_df_extended['avg_fee_rate_real']
median_eff_real = miner_df_extended['efficiency_real_fee'].median()
miner_df_extended['label_real_fee'] = (miner_df_extended['efficiency_real_fee'] > median_eff_real).astype(int)

feature_columns_extended = ['blocks_mined', 'avg_transactions', 'avg_volume',
                            'avg_fee', 'fee_volatility', 'avg_block_size',
                            'difficulty', 'profitability', 'age',
                            'median_fee_rate_real', 'fee_range_min_real',
                            'fee_range_max_real', 'avg_input_count', 'avg_output_count']
# note: avg_fee_rate_real deliberately excluded from features since it's now the label source

X = miner_df_extended[feature_columns_extended]
y = miner_df_extended['label_real_fee']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

model = MLPClassifier(
    hidden_layer_sizes=(64, 32, 16, 8), activation='relu', solver='adam', alpha=0.001,
    random_state=42, early_stopping=True, validation_fraction=0.1,
    n_iter_no_change=20, max_iter=100, batch_size=16
)
model.fit(X_train_scaled, y_train)

test_acc = model.score(X_test_scaled, y_test)
cv_scores = cross_val_score(model, X_train_scaled, y_train, cv=5)

print(f"Real-fee-rate-based label — test accuracy: {test_acc:.4f}")
print(f"Real-fee-rate-based label — CV mean: {cv_scores.mean():.4f}, std: {cv_scores.std():.4f}")
print(f"(Ethan's reported: test=0.95, CV=0.4875, std=0.0935)")

Real-fee-rate-based label — test accuracy: 0.5500
Real-fee-rate-based label — CV mean: 0.6375, std: 0.0829
(Ethan's reported: test=0.95, CV=0.4875, std=0.0935)


## Compare Kaggle vs BigQuery Raw Columns

In [31]:
from IPython.display import display

# ============================================================
# Kaggle vs BigQuery — raw column, aggregated feature, and
# full-pipeline comparison over the exact shared block range.
# Depends on preprocess() and train_and_test() already being
# defined earlier in this notebook.
# ============================================================

# --- Load both raw datasets ---
kaggle_df = pd.read_csv("../../data/kaggle_dataset.csv")
bq_df = pd.read_csv("../../data/real_bitcoin_blocks_raw.csv")

# --- Rename Kaggle's columns to BigQuery's naming convention ---
kaggle_adapted = kaggle_df.rename(columns={
    "height": "number",
    "tx_count": "transaction_count",
    "output_amount": "total_output_satoshis_excl_coinbase",
    "total_fees": "total_fee_satoshis",
})
# columns preprocess() expects to exist so it can drop them — never used downstream
for col in ["total_output_satoshis", "block_number", "tx_count_check", "bits"]:
    kaggle_adapted[col] = 0

# --- Restrict both to the exact shared block range ---
# BigQuery: 0-810908 (has genesis block, missing Kaggle's extra tip block)
# Kaggle:   1-810909 (missing genesis, has one extra tip block)
SHARED_MIN, SHARED_MAX = 1, 810908
bq_shared = bq_df[(bq_df["number"] >= SHARED_MIN) & (bq_df["number"] <= SHARED_MAX)].reset_index(drop=True)
kaggle_shared = kaggle_adapted[(kaggle_adapted["number"] >= SHARED_MIN) & (kaggle_adapted["number"] <= SHARED_MAX)].reset_index(drop=True)

print(f"Shared block range {SHARED_MIN}-{SHARED_MAX}: "
      f"BigQuery {len(bq_shared):,} rows, Kaggle {len(kaggle_shared):,} rows")

# --- Part 1: raw per-block column comparison ---
merged = bq_shared.merge(kaggle_shared, on="number", suffixes=("_bq", "_kaggle"))

raw_cols = {
    "size": "size",
    "transaction_count": "tx_count",
    "difficulty": "difficulty",
    "total_fee_satoshis": "total_fees",
    "total_output_satoshis_excl_coinbase": "output_amount"
}
raw_rows = []
for bq_col, kaggle_col in raw_cols.items():
    mism = (merged[f"{bq_col}_bq"] != merged[f"{bq_col}_kaggle"]).sum()
    max_diff = (merged[f"{bq_col}_bq"] - merged[f"{bq_col}_kaggle"]).abs().max()
    raw_rows.append({
        "BigQuery column": bq_col,
        "Kaggle column": kaggle_col,
        "Mismatched rows": f"{mism:,} / {len(merged):,}",
        "Max abs difference": max_diff,
    })

print("\nRaw per-block column comparison:")
display(pd.DataFrame(raw_rows))

# --- Part 2: per-miner aggregated feature comparison ---
miner_df_bq = preprocess(bq_shared, "total_output_satoshis_excl_coinbase", "ratio_of_means", "neighbor_gap")
miner_df_kaggle = preprocess(kaggle_shared, "total_output_satoshis_excl_coinbase", "ratio_of_means", "neighbor_gap")

def strip_tz(series):
    """BigQuery's timestamp is tz-aware (UTC); Kaggle's is tz-naive.
    Both represent the same UTC wall-clock time, so drop the tz label
    to make them directly comparable."""
    if pd.api.types.is_datetime64_any_dtype(series) and getattr(series.dt, "tz", None) is not None:
        return series.dt.tz_localize(None)
    return series

shared_feature_cols = [c for c in miner_df_bq.columns if c in miner_df_kaggle.columns and c != "miner_id"]

agg_rows = []
for col in shared_feature_cols:
    a = strip_tz(miner_df_bq[col])
    b = strip_tz(miner_df_kaggle[col])
    diff = (a - b).abs()
    if pd.api.types.is_timedelta64_dtype(diff):
        max_diff, unit = diff.dt.total_seconds().max(), "seconds"
    else:
        max_diff, unit = diff.max(), ""
    agg_rows.append({"Aggregated feature": col, "Max abs difference": max_diff, "Unit": unit})

print("\nPer-miner aggregated feature comparison:")
display(pd.DataFrame(agg_rows))

# --- Part 3: full-pipeline model result comparison ---
result_bq = train_and_test(miner_df_bq, seed=42, feature_set="full")
result_kaggle = train_and_test(miner_df_kaggle, seed=42, feature_set="full")

print("\nFull-pipeline result comparison (seed=42, full feature set):")
display(pd.DataFrame([{"Source": "BigQuery", **result_bq},
                       {"Source": "Kaggle", **result_kaggle}]))

Shared block range 1-810908: BigQuery 810,908 rows, Kaggle 810,908 rows

Raw per-block column comparison:


,BigQuery column,Kaggle column,Mismatched rows,Max abs difference
0,size,size,"0 / 810,908",0.000000
1,transaction_count,tx_count,"0 / 810,908",0.000000
2,difficulty,difficulty,"324,439 / 810,908",0.007812
3,total_fee_satoshis,total_fees,"0 / 810,908",0.000000
4,total_output_satoshis_excl_coinbase,output_amount,"0 / 810,908",0.000000



Per-miner aggregated feature comparison:


,Aggregated feature,Max abs difference,Unit
0,blocks_mined,0.000000e+00,
1,avg_transactions,0.000000e+00,
2,avg_volume,0.000000e+00,
3,avg_fee,0.000000e+00,
4,fee_volatility,0.000000e+00,
5,avg_block_size,0.000000e+00,
6,difficulty,9.765625e-04,
7,efficiency,0.000000e+00,
8,profitability,0.000000e+00,
9,last_block_id,0.000000e+00,



Full-pipeline result comparison (seed=42, full feature set):


,Source,test_accuracy,cv_mean,median_efficiency,min_efficiency,max_efficiency,corr_avg_transaction_efficiency,corr_avg_volume_efficiency,model_iterations
0,BigQuery,0.9,0.8125,0.107405,0.098501,0.114919,-0.000965,-0.982985,25
1,Kaggle,0.9,0.8125,0.107405,0.098501,0.114919,-0.000965,-0.982985,25


In [32]:
# Mismatch between interpretation of time in Kaggle (Unix Milliseconds) and BigQuery (Datetime)

print("Kaggle 'timestamp' dtype:", kaggle_df['timestamp'].dtype)
print(kaggle_df['timestamp'].head(3).tolist())
print()
print("BigQuery 'timestamp' dtype:", bq_df['timestamp'].dtype)
print(bq_df['timestamp'].head(3).tolist())
print()

kaggle_ts = pd.to_datetime(kaggle_shared['timestamp'])
bq_ts = pd.to_datetime(bq_shared['timestamp'])
print("Kaggle parsed range:  ", kaggle_ts.min(), "to", kaggle_ts.max())
print("BigQuery parsed range:", bq_ts.min(), "to", bq_ts.max())

Kaggle 'timestamp' dtype: int64
[1231469665000, 1231469744000, 1231470173000]

BigQuery 'timestamp' dtype: str
['2009-01-03 18:15:05+00:00', '2009-01-09 02:54:25+00:00', '2009-01-09 02:55:44+00:00']

Kaggle parsed range:   1970-01-01 00:20:31.469665 to 1970-01-01 00:28:16.599441
BigQuery parsed range: 2009-01-09 02:54:25+00:00 to 2023-10-06 13:37:21+00:00


In [33]:
kaggle_shared_fixed = kaggle_shared.copy()
kaggle_shared_fixed['timestamp'] = pd.to_datetime(kaggle_shared_fixed['timestamp'], unit='ms')

miner_df_kaggle_fixed = preprocess(kaggle_shared_fixed, "total_output_satoshis_excl_coinbase", "ratio_of_means", "neighbor_gap")

for col in ['last_block_time', 'age']:
    a = strip_tz(miner_df_bq[col])
    b = strip_tz(miner_df_kaggle_fixed[col])
    diff = (a - b).abs()
    if pd.api.types.is_timedelta64_dtype(diff):
        print(f"{col}: max abs diff = {diff.dt.total_seconds().max():.4f} seconds")
    else:
        print(f"{col}: max abs diff = {diff.max()}")

# Raw per-block timestamp comparison, for the Part 1 table
merged['timestamp_bq_parsed'] = pd.to_datetime(merged['timestamp_bq']).dt.tz_localize(None)
merged['timestamp_kaggle_parsed'] = pd.to_datetime(merged['timestamp_kaggle'], unit='ms')
ts_diff_sec = (merged['timestamp_bq_parsed'] - merged['timestamp_kaggle_parsed']).abs().dt.total_seconds()
print(f"timestamp: max abs diff = {ts_diff_sec.max():.6f} seconds, "
      f"rows with nonzero diff = {(ts_diff_sec > 0).sum():,} / {len(merged):,}")

last_block_time: max abs diff = 0.0000 seconds
age: max abs diff = 0.0
timestamp: max abs diff = 0.000000 seconds, rows with nonzero diff = 0 / 810,908


In [34]:
import sys, io, contextlib
import numpy as np
import pandas as pd

import warnings
from sklearn.exceptions import ConvergenceWarning
warnings.filterwarnings("ignore", category=ConvergenceWarning)

sys.path.insert(0, "../ethan_original")  # adjust the relative path to match where your notebook actually sits
from data_preprocessing import DataProcessor
from dnn_model import DNNModel

DATASETS = {
    "Kaggle":        "../../data/kaggle_dataset.csv",
    "Ethan cleaned": "../../data/bitcoin_for_weka.csv",
}
N_RUNS = 20  # matches the existing 20-run convention (§78 table)

def silent(fn, *a, **kw):
    """train()/evaluate()/load_data() print per-epoch logs and full
    classification reports on every call — suppress for a clean summary."""
    with contextlib.redirect_stdout(io.StringIO()):
        return fn(*a, **kw)

def build_miners_df(csv_path, fix_timestamp):
    dp = DataProcessor(csv_path)
    df = silent(dp.load_data)
    if fix_timestamp:
        # Pre-convert with the correct unit. create_miner_simulation()'s own
        # pd.to_datetime() call becomes a no-op on an already-datetime column,
        # so this reuses his unmodified method rather than reimplementing it.
        df['timestamp'] = pd.to_datetime(df['timestamp'], unit='ms')
    miners_df = dp.create_miner_simulation(df)
    return dp, miners_df

def run_comparison(csv_path, label):
    print(f"\n{'='*60}\n{label}\n{'='*60}")
    results = {}

    for fix in (False, True):
        tag = "fixed (unit='ms')" if fix else "buggy (default unit)"
        dp, miners_df = build_miners_df(csv_path, fix_timestamp=fix)

        print(f"\n--- {tag} ---")
        print(f"age: min={miners_df['age'].min():.6f}, "
              f"max={miners_df['age'].max():.6f}, "
              f"mean={miners_df['age'].mean():.6f}")

        X_train, X_test, y_train, y_test, features = silent(dp.prepare_training_data, miners_df)

        # Path B: cross_val_score is deterministic given a fixed seed (§88/§89) —
        # one run is enough for an exact comparison.
        X_full = np.vstack([X_train, X_test])
        y_full = np.concatenate([y_train, y_test])
        model_b = DNNModel(input_shape=X_train.shape[1])
        cv_scores = silent(model_b.cross_validate, X_full, y_full, cv=5)

        # Path A: his own train()/evaluate() is nondeterministic (§89) —
        # run N_RUNS times and compare distributions, not single draws.
        test_accs = []
        for _ in range(N_RUNS):
            model_a = DNNModel(input_shape=X_train.shape[1])
            silent(model_a.train, X_train, y_train, epochs=100, batch_size=16)
            test_accs.append(silent(model_a.evaluate, X_test, y_test))
        test_accs = np.array(test_accs)

        results[tag] = {
            "cv_mean": cv_scores.mean(),
            "test_acc_mean": test_accs.mean(),
            "test_acc_min": test_accs.min(),
            "test_acc_max": test_accs.max(),
        }

    print(f"\n>>> {label} summary:")
    print(pd.DataFrame(results).T.to_string(float_format=lambda v: f"{v:.4f}"))
    return results

all_results = {}
for name, path in DATASETS.items():
    all_results[name] = run_comparison(path, name)


Kaggle

--- buggy (default unit) ---
age: min=0.000000, max=0.002805, mean=0.000497

--- fixed (unit='ms') ---
age: min=0.000000, max=2805.000000, mean=497.260000

>>> Kaggle summary:
                      cv_mean  test_acc_mean  test_acc_min  test_acc_max
buggy (default unit)   0.4900         0.9750        0.9000        1.0000
fixed (unit='ms')      0.4900         0.9750        0.9000        1.0000

Ethan cleaned

--- buggy (default unit) ---
age: min=0.000000, max=0.002805, mean=0.000497

--- fixed (unit='ms') ---
age: min=0.000000, max=2805.000000, mean=497.260000

>>> Ethan cleaned summary:
                      cv_mean  test_acc_mean  test_acc_min  test_acc_max
buggy (default unit)   0.4900         0.9625        0.9000        1.0000
fixed (unit='ms')      0.4900         0.9675        0.9000        1.0000


In [35]:
# --- Load both raw datasets ---
kaggle_df = pd.read_csv("../../data/kaggle_dataset.csv")
bq_df = pd.read_csv("../../data/real_bitcoin_blocks_raw.csv")

# --- Restrict to the exact shared block range ---
kaggle_shared = kaggle_df[(kaggle_df["height"] >= 1) & (kaggle_df["height"] <= 810908)].reset_index(drop=True)
bq_shared = bq_df[(bq_df["number"] >= 1) & (bq_df["number"] <= 810908)].reset_index(drop=True)

# --- Direct comparison of the two 'difficulty' columns ---
merged = bq_shared.merge(
    kaggle_shared[["height", "difficulty"]],
    left_on="number", right_on="height",
    suffixes=("_bq", "_kaggle")
)

diff = (merged["difficulty_bq"] - merged["difficulty_kaggle"]).abs()
mismatches = (merged["difficulty_bq"] != merged["difficulty_kaggle"]).sum()

print(f"Mismatched rows: {mismatches:,} / {len(merged):,}")
print(f"Max absolute difference: {diff.max()}")
print(f"Mean absolute difference: {diff.mean()}")

print("\nWorst 5 mismatches:")
worst = diff.sort_values(ascending=False).index[:5]

result = merged.loc[worst, ["number", "difficulty_bq", "difficulty_kaggle"]].assign(abs_diff=diff.loc[worst])
print(result.to_string(float_format=lambda x: f'{x:,.8f}'))

Mismatched rows: 324,439 / 810,908
Max absolute difference: 0.0078125
Mean absolute difference: 0.0005878769745450721

Worst 5 mismatches:
        number               difficulty_bq           difficulty_kaggle   abs_diff
683799  683800 25,046,487,590,083.27734375 25,046,487,590,083.26953125 0.00781250
683798  683799 25,046,487,590,083.27734375 25,046,487,590,083.26953125 0.00781250
683797  683798 25,046,487,590,083.27734375 25,046,487,590,083.26953125 0.00781250
683796  683797 25,046,487,590,083.27734375 25,046,487,590,083.26953125 0.00781250
683795  683796 25,046,487,590,083.27734375 25,046,487,590,083.26953125 0.00781250


## feature columns had minor difference in definition, check effect

In [36]:
# check for feature definition difference effects

# Does row-position assignment (Ethan) match block_id-value assignment (yours)?
df_check = df.copy().reset_index(drop=True)          # default index = CSV load order
df_check = df_check.sort_values('number')            # Ethan's sort, original index labels kept

ethan_miner_id = df_check.index % 100
your_miner_id  = df_check['number'] % 100

mismatches = (ethan_miner_id.values != your_miner_id.values).sum()
print(f"Mismatched assignments: {mismatches} / {len(df_check)} "
      f"({mismatches/len(df_check)*100:.4f}%)")

# Does the epsilon choice ever change the binary label?
eff_ethan   = miner_df['avg_transactions'] / (miner_df['avg_volume'] + 1)
eff_project = miner_df['avg_transactions'] / (miner_df['avg_volume'] + 1e-9)

pct_diff = ((eff_ethan - eff_project).abs() / eff_project.abs()) * 100
print(f"Mean |% diff| in efficiency: {pct_diff.mean():.4f}%")
print(f"Max |% diff| in efficiency:  {pct_diff.max():.4f}%")

label_ethan   = (eff_ethan   > eff_ethan.median()).astype(int)
label_project = (eff_project > eff_project.median()).astype(int)
flips = (label_ethan != label_project).sum()
print(f"Miners whose binary label flips: {flips} / {len(miner_df)}")

Mismatched assignments: 0 / 810909 (0.0000%)
Mean |% diff| in efficiency: 0.0096%
Max |% diff| in efficiency:  0.0103%
Miners whose binary label flips: 0 / 100


## Export round-robin miner_df to csv for homogenization comparison

In [37]:
rr_df = preprocess(df, "total_output_satoshis", "ratio_of_means", "neighbor_gap")
rr_df.to_csv("../../data/round_robin_miner_df.csv", index=False)

### Make one round robin miner df with real fees just for comparison purposes

In [38]:
# Duplicate of preprocess() — only change is keeping total_fee_satoshis (real fee data)
# instead of dropping it, so we can compare it the same way build_miner_features() does.
# Original preprocess() is untouched.
def preprocess_with_real_fee(df, volume_column, efficiency_definition, age_definition):
    processed_df = df.rename(columns={
        'number': 'block_id',
        'transaction_count': 'n_transactions',
        volume_column: 'transaction_volume',
    })

    if volume_column == "total_output_satoshis":
        drop_columns = ['total_output_satoshis_excl_coinbase', 'block_number', 'tx_count_check', 'bits']
    elif volume_column == "total_output_satoshis_excl_coinbase":
        drop_columns = ['total_output_satoshis', 'block_number', 'tx_count_check', 'bits']

    processed_df = processed_df.drop(columns=drop_columns)

    satoshi_columns = ['transaction_volume']
    processed_df[satoshi_columns] = processed_df[satoshi_columns] / 1e8
    # total_fee_satoshis is deliberately NOT converted to BTC — build_miner_features()'s
    # avg_fee_real also stays in raw satoshis, so this keeps the two sides comparable.

    processed_df['block_efficiency'] = np.where(
        processed_df['transaction_volume'] > 0,
        processed_df['n_transactions'] / processed_df['transaction_volume'],
        np.nan
    )

    processed_df['miner_id'] = processed_df['block_id'] % 100
    processed_df['fee_proxy'] = processed_df['n_transactions'] * processed_df['transaction_volume']
    processed_df['timestamp'] = pd.to_datetime(processed_df['timestamp'])

    miner_df = processed_df.groupby('miner_id').agg(
        blocks_mined=('block_id', 'count'),
        avg_transactions=('n_transactions', 'mean'),
        avg_volume=('transaction_volume', 'mean'),
        avg_fee=('fee_proxy', 'mean'),
        fee_volatility=('fee_proxy', 'std'),
        avg_block_size=('size', 'mean'),
        difficulty=('difficulty', 'mean'),
        efficiency=('block_efficiency', 'mean'),
        profitability=('fee_proxy', 'sum'),
        avg_fee_real=('total_fee_satoshis', 'mean'),          # NEW
        fee_volatility_real=('total_fee_satoshis', 'std'),    # NEW
        profitability_real=('total_fee_satoshis', 'sum'),     # NEW
        last_block_id=('block_id', 'max'),
        last_block_time=('timestamp', 'max')
    ).reset_index()

    miner_df['profitability'] = miner_df['profitability'] / (miner_df['blocks_mined'] + 1)
    miner_df['profitability_real'] = miner_df['profitability_real'] / (miner_df['blocks_mined'] + 1)  # NEW

    if age_definition == "distance_from_tip":
        miner_df['age'] = processed_df['timestamp'].max() - miner_df['last_block_time']
        miner_df['age'] = miner_df['age'].dt.total_seconds()
    elif age_definition == "neighbor_gap":
        sorted_miners = miner_df.sort_values('last_block_time').reset_index(drop=True)
        sorted_miners['age'] = sorted_miners['last_block_time'].diff().dt.total_seconds()
        sorted_miners['age'] = sorted_miners['age'].fillna(sorted_miners['age'].median())
        miner_df = miner_df.drop(columns=['age'], errors='ignore').merge(
            sorted_miners[['miner_id', 'age']], on='miner_id'
        )

    if efficiency_definition == "ratio_of_means":
        miner_df['efficiency'] = miner_df['avg_transactions'] / (miner_df['avg_volume'] + 1e-9)
    elif efficiency_definition == "mean_of_ratios":
        pass

    median_efficiency = miner_df['efficiency'].median()
    miner_df['label'] = (miner_df['efficiency'] > median_efficiency).astype(int)

    return miner_df


rr_df_fee = preprocess_with_real_fee(df, "total_output_satoshis", "ratio_of_means", "neighbor_gap")
rr_df_fee.to_csv("../../data/round_robin_miner_df_with_real_fee.csv", index=False)  # separate file, doesn't touch the original